In [27]:
import pandas as pd
import numpy as np
import matplotlib
import matplotlib.pyplot as plt
import glob
import os
import cmath 
import json
import seaborn as sns
from collections import Counter
from scipy.stats import linregress
import PyQt5
# %matplotlib qt
import mne
from tqdm import tqdm
from scipy.io import loadmat

In [28]:
SUBJS = ["07", "09", "10", "11", "12"]
SUBJS_NOT_GOOD = ["01", "02", "03", "05", "06", "08"]

# **Data Loading and informations**

In [41]:
for subject in SUBJS:
    print("printing subject ", subject, "...")
    epochs = mne.read_epochs(f"/Volumes/My Passport/Data/derivatives/mne-bids-pipeline/sub-{subject}/meg/sub-{subject}_task-FRINT_proc-clean_epo.fif", preload=True)
    metadatas = pd.read_csv(f"/Volumes/My Passport/Data comportement/data clean behavior MEG/FRINT_MEG_S{subject}.csv", index_col = 0)
    
    metadatas.columns = metadatas.columns.str.strip()
    
    
    df_filtered = metadatas.copy()
    # Exemple: sélection des colonnes rotangle_1 à rotangle_7
    rotangle_cols = [f'rotangle_{i}' for i in range(1, 8)]
    rotation_cols = [f'rotations_{i}' for i in range(1, 8)]
    presangle_cols = [f'presangle_{i}' for i in range(1, 8)]
    df_filtered['presented_angles'] = df_filtered[presangle_cols].apply(lambda row: row.dropna().tolist(), axis=1)
    df_filtered['expected_angles'] = df_filtered[rotangle_cols].apply(lambda row: row.dropna().tolist(), axis=1)
    df_filtered['stimulus_rotations'] = df_filtered[rotation_cols].apply(lambda row: row.dropna().tolist(), axis=1)
    df_filtered["expected_angles_wo_rot"] = df_filtered["presavg"]
    df_filtered["length of sequence"] = df_filtered["Seqlen"]
    df_filtered = df_filtered.drop(columns=rotangle_cols + rotation_cols + presangle_cols)
    df_filtered = df_filtered.drop(columns=["optimal", "presavg", "Seqlen"])
    df_filtered.reset_index(drop=True, inplace=True)
    df_filtered = df_filtered.dropna(subset=["final_angle"])
    
    df_radian = df_filtered.copy().drop(columns=["final_angle", "GenMean", "target", "expected_angles_wo_rot"])

    df_radian['final_angle'] = np.deg2rad(df_filtered['final_angle'])
    df_radian["target"] = np.deg2rad(df_filtered['target'])
    df_radian["GenMean"] = np.deg2rad(df_filtered['GenMean'])
    df_radian["expected_angles_wo_rot"] = np.deg2rad(df_filtered['expected_angles_wo_rot'])
    df_radian['stimulus_rotations_rad'] = df_filtered['stimulus_rotations'].apply(
        lambda lst: np.deg2rad(np.array(lst, dtype=float)))
    df_radian['stimulus_rotations_degrees'] = df_filtered['stimulus_rotations']
    df_radian['presented_angles_rad'] = df_filtered['presented_angles'].apply(
        lambda lst: np.deg2rad(np.array(lst, dtype=float)))
    df_radian['expected_angles_rad'] = df_filtered['expected_angles'].apply(
        lambda lst: np.deg2rad(np.array(lst, dtype=float)))

    df_radian["mean_rotation"] = df_radian["stimulus_rotations_rad"].apply(np.mean) 
    metadatas_clean = df_radian.copy()
    
    if float(subject)%3 == 0 :
        color_mapping = {np.deg2rad(0): 'blue',  np.deg2rad(45): 'green', np.deg2rad(-45): 'red'}
    elif float(subject)%3 == 1 :
        color_mapping = {np.deg2rad(0): 'red',  np.deg2rad(45): 'blue', np.deg2rad(-45): 'green'}
    elif float(subject)%3 == 2 : 
        color_mapping = {np.deg2rad(0): 'green',  np.deg2rad(45): 'red', np.deg2rad(-45): 'blue'}
    
    
    colors = []
    colors_extended = []
    for i in range(len(metadatas_clean)) : 
        colors_one_extended = []
        colors_one = []
        for x in metadatas_clean["stimulus_rotations_rad"].iloc[i]:
            colors_one.append(color_mapping[x])
            colors_one_extended.extend(color_mapping[x])
        colors.append(colors_one)
        colors_extended.extend(colors_one)

    metadatas_clean["color_seen"] = colors
    
    
    data = loadmat(f"/Volumes/My Passport/Data comportement/M5_MEG_FIT/{subject}_M5_VBMC.mat")
    out = data["out_vbmc"][0,0]
    average = out["xavg"].squeeze()
    average_deg = ((np.array(average) *180/np.pi) )  # convert the angles in degrees
    d_rot_p = average_deg[2] 
    d_rot_m = average_deg[3]
    
    
    block_extended = []
    rotation_radians_extended = []
    rotation_degrees_extended = []
    expected_angles_extended = []
    expected_angles_rad_extended = []

    presented_angles_rad_extended = []
    presented_angles_extended = []
    fitted_value = []

    count = 0
    for i in range(len(metadatas_clean)) :
        for element in range(len(metadatas_clean["stimulus_rotations"].iloc[i])) :
            block_extended.append(metadatas_clean["Block"].iloc[i])
            presented_angles_extended.append(metadatas_clean["presented_angles"].iloc[i][element])
            presented_angles_rad_extended.append(metadatas_clean["presented_angles_rad"].iloc[i][element]%np.pi)
            rotation_radians_extended.append(metadatas_clean["stimulus_rotations"].iloc[i][element]%np.pi)
            rotation_degrees_extended.append(metadatas_clean["stimulus_rotations_degrees"].iloc[i][element])
            expected_angles_extended.append(metadatas_clean["expected_angles"].iloc[i][element])   
            expected_angles_rad_extended.append(metadatas_clean["expected_angles_rad"].iloc[i][element]%np.pi)  

            count +=1 
    
    fitted_expected_angles_extended = []
    fitted_expected_angles_rad_extended = []
    for rotation, presented_angle in zip(rotation_degrees_extended, presented_angles_extended) : 
        if rotation == 45.0 :
            fitted_expected_angles_extended.append(presented_angle + d_rot_p)
            fitted_expected_angles_rad_extended.append(np.deg2rad(presented_angle + d_rot_p))
            fitted_value.append(d_rot_p)
        elif rotation == -45.0 :
            fitted_expected_angles_extended.append(presented_angle + d_rot_m)
            fitted_expected_angles_rad_extended.append(np.deg2rad(presented_angle + d_rot_m))
            fitted_value.append(d_rot_m)

        elif rotation == 0.0 :
            fitted_expected_angles_extended.append(presented_angle)
            fitted_expected_angles_rad_extended.append(np.deg2rad(presented_angle))
            fitted_value.append(0)

    
    metadata_to_csv = pd.DataFrame()
    metadata_to_csv["stim_color"] = colors_extended
    metadata_to_csv["presented_angles_rad"] = presented_angles_rad_extended
    metadata_to_csv["block"] = block_extended
    metadata_to_csv["rotation_in_radian"] = rotation_radians_extended
    metadata_to_csv["rotation_in_degrees"] = rotation_degrees_extended
    metadata_to_csv["fitted_value_deg"] = fitted_value
    metadata_to_csv["expected_angles"] = expected_angles_extended
    metadata_to_csv["expected_angles_rad"] = expected_angles_rad_extended
    metadata_to_csv["presented_angles"] = presented_angles_extended
    metadata_to_csv["fitted_expected_angles"] = fitted_expected_angles_extended
    metadata_to_csv["fitted_expected_angles_rad"] = fitted_expected_angles_rad_extended
    
    
    metadata_to_csv.to_csv(f'/Volumes/My Passport/Data comportement/data clean behavior MEG/metadata_for_epochs/FRINT_MEG_S{subject}_cleaned_metadata.csv', index=False)

printing subject  07 ...
Reading /Volumes/My Passport/Data/derivatives/mne-bids-pipeline/sub-07/meg/sub-07_task-FRINT_proc-clean_epo.fif ...
    Reading extended channel information
    Read a total of 2 projection items:
        meg-ECG--0.500-0.500)-PCA-01 (1 x 306) active
        meg-EOG--0.500-0.500)-PCA-01 (1 x 306) active
    Found the data of interest:
        t =    -200.00 ...    1500.00 ms
        0 CTF compensation matrices available
Adding metadata with 14 columns
1440 matching events found
No baseline correction applied
Created an SSP operator (subspace dimension = 2)
2 projection items activated
printing subject  09 ...
Reading /Volumes/My Passport/Data/derivatives/mne-bids-pipeline/sub-09/meg/sub-09_task-FRINT_proc-clean_epo.fif ...
    Reading extended channel information
    Read a total of 2 projection items:
        meg-ECG--0.500-0.500)-PCA-01 (1 x 306) active
        meg-EOG--0.500-0.500)-PCA-01 (1 x 306) active
    Found the data of interest:
        t =    -200.0